In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 1 : 라이브러리 설치 확인 (최초 1회만 실행)
# ─────────────────────────────────────────────────────────────
import subprocess, sys

required = ['yfinance', 'pandas', 'numpy', 'matplotlib']
for pkg in required:
    try:
        __import__(pkg)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('✅ 모든 라이브러리 준비 완료')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 2 : 모듈 import
# ─────────────────────────────────────────────────────────────
import yfinance as yf
import pandas as pd

from step1_financials import analyze_financials
from step2_charts     import analyze_charts
from step3_news       import analyze_news

print('✅ 모듈 로드 완료')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 3 : S&P 500 기준선 — 프로그램 시작 시 1회만 로드
#           (같은 세션에서 여러 종목 분석 시 재사용 — API 절약)
# ─────────────────────────────────────────────────────────────
print('📡 S&P 500 기준선 로딩 중...')
try:
    SPX = yf.Ticker('^GSPC').history(period='5y')
    if SPX.index.tz is not None:
        SPX.index = SPX.index.tz_convert(None)
    print(f'✅ S&P 500 로드 완료  ({SPX.index[0].date()} ~ {SPX.index[-1].date()})')
except Exception as e:
    SPX = None
    print(f'⚠️  S&P 500 로드 실패: {e}  (차트에서 비교선 생략됨)')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4 : 종목 분석 실행  ← 티커만 바꿔서 반복 실행
#
# API 호출 구조 (종목당 총 5회)
#   1. stock.info          — 재무 전체 (STEP 1·3 공용)
#   2. stock.history(5y)   — 주가 5년치 (STEP 2 슬라이싱 재사용)
#   3. stock.news          — 최근 뉴스
#   4. stock.dividends     — 배당 이력
#   5. stock.earnings_dates — EPS 분기 이력
#   + S&P500은 CELL 3에서 캐시 (추가 호출 없음)
# ─────────────────────────────────────────────────────────────
TICKER = input('🔍 분석할 종목 티커 입력 (예: AAPL, TSLA, NVDA): ').upper().strip()

if not TICKER:
    print('티커를 입력해주세요.')
else:
    sep = '=' * 60
    print(f'\n{sep}')
    print(f'  [{TICKER}] 종합 분석 시작  |  API 호출 최소화 모드')
    print(f'{sep}')

    stock = yf.Ticker(TICKER)

    # 1/5: 재무 데이터
    print('  1/5  재무 데이터 수집 중...')
    info = stock.info

    if not info or not (info.get('currentPrice') or info.get('regularMarketPrice')):
        print(f'\n❌ [{TICKER}] 유효하지 않은 티커입니다. 철자를 확인해주세요.')
    else:
        # 2/5: 주가 5년치
        print('  2/5  주가 히스토리 수집 중 (5년)...')
        history = stock.history(period='5y')
        if history.index.tz is not None:
            history.index = history.index.tz_convert(None)

        # 3/5: 뉴스
        print('  3/5  최근 뉴스 수집 중...')
        try:
            raw_news = stock.news
        except Exception:
            raw_news = []

        # 4/5: 배당 이력
        print('  4/5  배당 이력 수집 중...')
        try:
            raw_dividends = stock.dividends
            if raw_dividends.index.tz is not None:
                raw_dividends.index = raw_dividends.index.tz_convert(None)
        except Exception:
            raw_dividends = None

        # 5/5: EPS 분기 이력
        print('  5/5  EPS 실적 이력 수집 중...')
        try:
            raw_eps = stock.earnings_dates
        except Exception:
            raw_eps = None

        print(f'\n✅ 데이터 수집 완료  →  분석 출력 시작\n')

        # ── STEP 1: 재무 지표 ───────────────────────────────
        analyze_financials(TICKER, info)

        # ── STEP 2: 기술적 차트 (SPX = CELL 3 캐시)
        analyze_charts(TICKER, history, SPX)

        # ── STEP 3: 뉴스·배당·EPS·거시경제·섹터 리스크·시나리오
        analyze_news(TICKER, info, raw_news,
                     raw_dividends=raw_dividends,
                     raw_eps=raw_eps,
                     stock=stock)